In [ ]:
# 0) setup: session clock, embedded code, environment
import base64, glob, hashlib, io, json, os, subprocess, tarfile, threading, time
T0 = time.time()
HOURS = 11.0                     # training stops this many hours after this cell starts
DEADLINE = T0 + HOURS * 3600
LIMIT = T0 + 11.6 * 3600            # nothing optional starts after this (Kaggle stops at 12 h)
CODE, OUT = '/kaggle/working/code', '/kaggle/working/out'
B64 = '...'
raw = base64.b64decode(B64)
assert hashlib.sha256(raw).hexdigest() == 'e0e35e17357d5956418051b7ee8c818ec9055e1e9618ec2e49e944ebb605a969'
tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz').extractall(CODE)
os.makedirs(OUT, exist_ok=True)
print('code sha256 e0e35e17357d5956418051b7ee8c818ec9055e1e9618ec2e49e944ebb605a969 from git 8003f49+local changes:', sorted(os.listdir(CODE)))


def run(cmd, timeout=None, env=None):
    """Run a shell command in CODE, stream its output, return (exit code, output). A timeout kills it."""
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=dict(os.environ, **(env or {})), start_new_session=True)
    timer = threading.Timer(timeout, lambda: os.killpg(p.pid, 9)) if timeout else None
    if timer:
        timer.start()
    out = []
    for line in p.stdout:
        print(line, end='', flush=True)
        out.append(line)
    rc = p.wait()
    if timer:
        timer.cancel()
    print(f'[exit {rc} after {(time.time() - T0) / 3600:.2f} h of the session]', flush=True)
    return rc, ''.join(out)


run('nvidia-smi --query-gpu=index,name,memory.total --format=csv; free -g | head -2; nproc; '
    'python -c "import torch; print(torch.__version__, torch.cuda.device_count())"; find /kaggle/input -maxdepth 5 | head -20')
INIT = sorted(glob.glob('/kaggle/input/**/best_ema.pt', recursive=True))[0]
COMMON = '--mat /kaggle/input --cache /tmp/chikusei_crop.npy'
NGPU = int(subprocess.check_output('nvidia-smi -L | wc -l', shell=True))
ENV = {'NCCL_P2P_DISABLE': '1', 'TORCH_NCCL_ASYNC_ERROR_HANDLING': '1', 'OMP_NUM_THREADS': '2'}
print('warm start:', INIT, '| GPUs:', NGPU, '| deadline in', HOURS, 'h')


In [ ]:
# 1) normalised 2048x2048x128 crop -> /tmp cache (shared by every later step) + operator / metric self-checks
rc, _ = run('python -c "import time; t = time.time(); from hsifuse.data import find_mat, load_chikusei; '
            'c = load_chikusei(find_mat(\'/kaggle/input\'), \'/tmp/chikusei_crop.npy\'); '
            'print(c.shape, c.dtype, round(time.time() - t), \'s\')"')
assert rc == 0, 'data cache failed'
run('python selfcheck.py')


In [ ]:
# 2) the warm-start checkpoint exactly as trained (zero-padded physics): full metrics incl. SSIM_psrt, Q2n, SCC
run(f'python test.py {COMMON} --ckpt {INIT} --pad zeros --out {OUT}/first_run')


In [ ]:
# 3) speed probe: DDP on all GPUs vs one GPU (identical training step). A hung / failing DDP falls back to one GPU.
import re
TRAIN = f'train.py {COMMON} --init_ckpt {INIT} --pad reflect --bs 8 --lr 1.5e-4 --warmup 1000 --w_sam 0.05 --w_ssim 0.1 --eval_every 2000 --log_every 500'
rate = {}
if NGPU > 1:
    rc, out = run(f'torchrun --standalone --nproc_per_node {NGPU} {TRAIN} --probe 150 --out /tmp/probe', timeout=600, env=ENV)
    m = re.search(r'PROBE ([0-9.]+) samples/s', out)
    rate[NGPU] = float(m.group(1)) if rc == 0 and m else 0.0
rc, out = run(f'python {TRAIN} --probe 150 --out /tmp/probe', timeout=600, env=ENV)
m = re.search(r'PROBE ([0-9.]+) samples/s', out)
rate[1] = float(m.group(1)) if rc == 0 and m else 0.0
USE_DDP = NGPU > 1 and rate[NGPU] > 1.25 * rate[1]
print('samples/s:', rate, '-> DDP' if USE_DDP else '-> one GPU')


In [ ]:
# 4) fine-tune until the deadline (checkpoints every eval: last.pt, best_ema.pt). If it dies, resume on one GPU.
LOG = f'{OUT}/puformer'
launch = f'torchrun --standalone --nproc_per_node {NGPU} ' if USE_DDP else 'python '
rc, _ = run(f'{launch}{TRAIN} --deadline {DEADLINE} --out {LOG}', env=ENV)
if rc != 0 and os.path.exists(f'{LOG}/last.pt') and time.time() < DEADLINE - 1200:
    print('training exited with', rc, '- resuming on one GPU until the deadline')
    rc, _ = run(f'python {TRAIN} --sched time --deadline {DEADLINE} --out {LOG}', env=ENV)
if not os.path.exists(f'{LOG}/results.json') and os.path.exists(f'{LOG}/best_ema.pt'):
    run(f'python test.py {COMMON} --ckpt {LOG}/best_ema.pt --pad reflect --out {LOG}/final_test')


In [ ]:
# 5) robustness study (blur / SRF mismatch with operator swap, noise, consistency), only if time is left
if os.path.exists(f'{OUT}/puformer/best_ema.pt') and time.time() < LIMIT - 1800:
    run(f'python eval_gaps.py {COMMON} --ckpt {OUT}/puformer/best_ema.pt --pad reflect --out {OUT}/puformer',
        timeout=max(60, LIMIT - time.time() - 300))


In [ ]:
# 6) summary
for f in sorted(glob.glob(f'{OUT}/**/results.json', recursive=True)):
    r = json.load(open(f))
    print(f, {k: r[k] for k in ('iters', 'epochs', 'gpus', 'best_val_PSNR', 'train_hours') if k in r})
    for k in ('val', 'test', 'test_tta', 'consistency', 'consistency_tta'):
        if k in r:
            print(' ', k, {m: round(v, 4) for m, v in r[k].items()})
run(f'ls -la {OUT}/*; du -sh {OUT}')
